# Declaring an estimand

An `Estimand` is a complete transferability key. Two quantities are the same quantity iff all
eight facets match:

| Facet | Field(s) |
|---|---|
| `quantity` | `quantity` — contrast, marginal, ratio, elasticity, area |
| `intervention` | `treatment`, `intervention`, `reference` |
| `outcome` | `outcome` |
| `population` | `population` |
| `window` | `window` |
| `level` | `level` — unit of analysis + interference model |
| `conditioning` | `conditioning` |
| `dimension` | `dimension` — derived, asserted against the declaration |

Nothing about an estimand may be implicit in the model that produced it. In the parent repo
most of these lived in a variable name.

In [ ]:
from axiom.core import D, DimensionError, Intervention, Outcome, Population, Spec, TimeWindow, Treatment
from axiom.estimands import FACETS, Estimand, Facet, Level, Quantity, QuantityKind, derived_dimension

In [ ]:
fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD")
yield_total = Outcome(name="yield_total", dimension=D.outcome, unit="kg", aggregation="sum")
north = Population(name="north", strata={"soil": {"clay": 0.3, "loam": 0.7}})
season = TimeWindow(start=0, stop=8, basis="cumulative")

lift = Estimand(
    name="lift_at_100",
    quantity=Quantity(kind="contrast"),
    treatment=fertilizer,
    intervention=Intervention(doses={"fertilizer": 100.0}, version="granular"),
    reference=Intervention(doses={"fertilizer": 0.0}, version="granular"),
    outcome=yield_total,
    population=north,
    window=season,
    level=Level(unit="cluster", interference="none"),
    conditioning=(),
    dimension=D.outcome,
    description="season-total yield lift from 100 USD of granular fertilizer vs none, north region",
)
print(lift.content_hash()[:16])

## The derived dimension is asserted

`derived_dimension(kind, outcome, dose)` is the table; the class refuses a declaration that
disagrees with it.

In [ ]:
kinds: list[QuantityKind] = ["contrast", "marginal", "ratio", "elasticity", "area"]
for kind in kinds:
    print(f"{kind:11s} -> {derived_dimension(kind, D.outcome, D.currency)}")

try:
    lift.model_copy(update={"dimension": D.currency}).model_validate(lift.model_copy(update={"dimension": D.currency}).to_dict())
except DimensionError as e:
    print("refused:", e)

## Other functionals

A marginal effect needs no reference; a ratio does. The intervention must set the treatment
the quantity is about.

In [ ]:
mroi = lift.model_copy(update={
    "name": "marginal_at_100",
    "quantity": Quantity(kind="marginal"),
    "reference": None,
    "dimension": D.outcome / D.currency,
})
print(mroi.name, mroi.dimension)

try:
    Estimand.model_validate({**lift.to_dict(), "quantity": {"kind": "ratio", "scale": "natural"}, "reference": None, "dimension": (D.outcome / D.currency).to_dict()})
except ValueError as e:
    print("refused:", str(e).splitlines()[-1].strip())

## Facets are inspectable

`FACETS` is the closed list; `facet()` returns what `transfer_to` compares; `differing_facets`
is the typed diff between two declarations.

In [ ]:
print(FACETS)
f: Facet = "intervention"
print(lift.facet(f))
print(lift.differing_facets(mroi))

## Serialization

An estimand is a `Spec`: it round-trips and its hash is its identity. Two producers — an
experiment, a fitted surface, a meta-analysis — that declare the same facets produce the same
hash, which is exactly what makes their results comparable.

In [ ]:
print(Spec.from_json(lift.to_json()) == lift)
print(lift.to_json(indent=2)[:300], "...")